# Day 072 · 反转因子 · 中国版
**Reversal** · 阶段 P3 · 数据与因子工程

> 上一节动量因子追强者恒强,这一节我们讲一个跟它正好唱反调的因子:反转因子。它的想法是,在很短的时间里——比如一周——跌得最惨的票,反而常常会反弹一下;涨得最猛的,反而会回落一下。也就是「物极必反、超跌反弹」。你可能会问,这不是跟上一节的动量矛盾吗?一个说强者恒强、一个说跌多了会反弹,到底听谁的?答案很有意思:它们管的时间尺度不一样。短期(一两周)看反转,中期(半年到一年)看动量,同一批票,短期要买输家、中期要买赢家,方向正好相反。不过这一节我得提前跟你说句实话:短期反转在A股的大盘股上其实很弱、很不稳定,主要在小盘、散户密集的票上才看得出来。我们会用真实数据,诚实地告诉你它到底有多强、又弱在哪。

---

### 关于「中国版」

本 notebook 是为**国内学员**优化的版本:
- 数据源用 **akshare**(国内可访问、零 VPN、免注册),取代了视频里的 yfinance
- 标的尽量保持原意:美股 ETF→A 股 ETF / 国际公司→A 股龙头
- 所讲的**概念和方法 100% 一致**,但**具体数字可能与视频里略有差异**(因为是不同时间窗 / 不同标的)
- 一般情况国内 `pip install akshare` 即可,无需 token / VPN

**课件生成日期:** 2026-06-27  ·  **建议学习时长:** 18 分钟

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有必需的 Python 包(含 `akshare`),缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续


In [1]:
# === 环境自检 + 自动安装(运行此单元格即可)===
import importlib, subprocess, sys, os

REQUIRED = ["akshare", "baostock", "matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels"]
PIP_NAME = {"sklearn":"scikit-learn","cv2":"opencv-python","PIL":"Pillow","bs4":"beautifulsoup4","yaml":"PyYAML"}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))
if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置 ===
import matplotlib, matplotlib.pyplot as plt, matplotlib.font_manager as fm
CJK = ["/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
       "C:/Windows/Fonts/msyh.ttc","C:/Windows/Fonts/simhei.ttf",
       "/System/Library/Fonts/PingFang.ttc","/System/Library/Fonts/STHeiti Medium.ttc"]
for p in CJK:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP","Microsoft YaHei","PingFang SC","SimHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪")


✓ 所有 9 个必需库已就绪
✓ 中文字体已加载:NotoSansCJK-Regular.ttc
✓ 环境就绪


## 🔌 第二步:加载国内数据助手

下面这一格是**工具函数**(可以折叠,不需要修改)。它把 `yfinance` 风格的 ticker(如 `600519.SS`)自动路由到对应的 akshare 接口,提供 `get_close(ticker)` 和 `get_close_multi(tickers_dict)` 两个函数。

In [2]:
# === 国内数据源助手(akshare 后端,不需要 VPN)===
# 这一格是工具函数,可以折叠,不需要修改。
# 它把 yfinance 风格的 ticker(如 "600519.SS" / "0700.HK" / "AAPL" / "BTC-USD")
# 自动路由到对应的 akshare 接口,统一返回 yfinance 风格的 Close DataFrame。

import re
from datetime import datetime, timedelta
import pandas as pd
import akshare as ak

_TICKER_MAP = {
    "^GSPC": ("us_index_sina", ".INX"),
    "^DJI":  ("us_index_sina", ".DJI"),
    "^IXIC": ("us_index_sina", ".IXIC"),
    "GC=F":  ("foreign_futures", "GC"),
    "SI=F":  ("foreign_futures", "SI"),
    "CL=F":  ("foreign_futures", "CL"),
    "BTC-USD": ("crypto", "BTC"),
    "ETH-USD": ("crypto", "ETH"),
}

def _retry(fn, *args, _retries=4, _wait=1.5, **kwargs):
    """akshare 上游(东方财富/新浪/Binance)偶有 RemoteDisconnected / Timeout,自动重试 4 次。
    2026-05-11 加:用户跑 cn notebook 拉 002594.SZ 时上游断连 → 整节卡死。
    每次重试间隔 _wait 秒(指数退避 1x → 1.5x → 2.25x)。
    """
    import time as _t
    last_err = None
    wait = _wait
    for i in range(_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            name = type(e).__name__
            if i == _retries - 1:
                print(f"  ✗ {getattr(fn,'__name__',str(fn))} 第 {i+1} 次仍失败({name}),放弃")
                raise
            print(f"  ⚠ {getattr(fn,'__name__',str(fn))} 第 {i+1} 次失败({name}),{wait:.1f}s 后重试")
            _t.sleep(wait)
            wait *= 1.5

def _parse_period(period):
    end = datetime.today()
    m = re.match(r"^(\d+)\s*(y|mo|d|w)$", period.lower().strip())
    days = 365 * 3 if not m else int(m.group(1)) * {"y":365,"mo":30,"w":7,"d":1}[m.group(2)]
    return (end - timedelta(days=days+30)).strftime("%Y%m%d"), end.strftime("%Y%m%d")

def _classify(ticker):
    t = ticker.strip()
    if t in _TICKER_MAP: return _TICKER_MAP[t]
    if t.endswith((".SS",".SH",".SZ")):
        code = t.split(".")[0]
        if code.startswith(("51","159","58")) or code in ("510300","510500","510050","511010","513100"):
            return ("a_etf", code)
        if code in ("000300","000016","000905","000852","000001"):
            return ("a_index", code)
        return ("a_stock", code)
    if t.endswith(".HK"):
        return ("hk", t.split(".")[0].zfill(5))
    return ("us", t)

def _norm(df, dc, cc):
    out = df[[dc, cc]].copy()
    out[dc] = pd.to_datetime(out[dc])
    return out.set_index(dc).sort_index()[cc].astype(float).rename("Close")

def get_close(ticker, period="3y"):
    """返回某标的 Close 价格 series。后端 akshare,中国可访问。
    所有 ak.* 调用都过 _retry(4 次,指数退避)— 防东方财富/新浪上游瞬时断连。
    """
    start, end = _parse_period(period)
    kind, sym = _classify(ticker)
    if kind == "a_stock":
        return _norm(_retry(ak.stock_zh_a_hist, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "a_etf":
        return _norm(_retry(ak.fund_etf_hist_em, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "a_index":
        idx_map = {"000300":"sh000300","000016":"sh000016","000905":"sh000905","000852":"sh000852","000001":"sh000001"}
        s = _norm(_retry(ak.stock_zh_index_daily_em, symbol=idx_map.get(sym, f"sh{sym}")), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "hk":
        return _norm(_retry(ak.stock_hk_hist, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "us":
        s = _norm(_retry(ak.stock_us_daily, symbol=sym, adjust="qfq"), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "us_index_sina":
        s = _norm(_retry(ak.index_us_stock_sina, symbol=sym), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "foreign_futures":
        s = _norm(_retry(ak.futures_foreign_hist, symbol=sym), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "crypto":
        import requests as _rq
        def _binance():
            r = _rq.get("https://api.binance.com/api/v3/klines",
                        params={"symbol": f"{sym}USDT", "interval": "1d", "limit": 1000}, timeout=15)
            r.raise_for_status()
            return r.json()
        klines = _retry(_binance)
        df = pd.DataFrame(klines, columns=["open_time","open","high","low","close","volume",
                                            "close_time","qav","trades","tbb","tbq","ignore"])
        df["date"] = pd.to_datetime(df["open_time"], unit="ms")
        df["close"] = df["close"].astype(float)
        s = df.set_index("date").sort_index()["close"].rename("Close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    raise ValueError(f"unsupported ticker: {ticker}")

def get_close_multi(tickers, period="3y"):
    """批量取 Close,返回 DataFrame,列名是 tickers dict 的 key(中文名),按交集日期对齐。"""
    series = {name: get_close(t, period=period) for name, t in tickers.items()}
    return pd.concat(series, axis=1).sort_index()

print("✓ cn_data 助手已加载 — 用 get_close(ticker) / get_close_multi(tickers_dict) 拉数据")


✓ cn_data 助手已加载 — 用 get_close(ticker) / get_close_multi(tickers_dict) 拉数据


## 学习目标

- 搞懂反转因子在赌什么:短期内跌最惨的超跌反弹、涨最猛的回落,赌「物极必反」
- 理解反转为什么存在:散户的过度反应,加上有人愿意接盘提供流动性、赚个反弹钱
- 彻底搞清反转和动量的关系:短期买输家、中期买赢家,同一批票方向正好相反
- 认清A股短期反转的真相:在大盘股很弱很不稳,只在小盘散户票才略明显
- 了解反转的更多形态:隔夜反转、日内反转,以及成交量大时反转更显著

## 历史背景:老王听说「超跌反弹」能抄底,专挑跌最惨的大盘白马抄,抄了一年发现根本不灵,后来才懂:反转这事挑错了票池

老王听一个朋友说,股市里有个规律叫超跌反弹——跌得最狠的票,过几天总会弹回来一点,专抄这种,稳赚反弹钱。老王一听有道理,就盯着那些跌得多的大盘蓝筹、白马股抄底,跌得越惨他越买。可抄了大半年,他发现根本不是那么回事:有些大白马跌下去之后,不但不反弹,反而沿着趋势一路阴跌,他抄一次套一次。老王很困惑,明明书上说短期会反转啊。后来他请教一个做量化的朋友才明白:短期反转这个效应,在流动性特别好的大盘股上很弱,因为大盘股价格已经被无数机构盯着、定价很有效,不太会过度反应;它主要发生在小盘、换手率高、散户扎堆的票上——这些票散户情绪化、容易追涨杀跌、过度反应,砸狠了之后才容易被错杀、被买回来。老王挑错了池子,难怪不灵。反转因子不是不存在,而是它有很挑剔的适用范围。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. 1. 反转因子:短期内,物极必反

反转因子的想法和动量正相反:它赌的是,在很短的时间里(常说一到4 周),跌得最惨的票会超跌反弹、涨得最猛的会见好就收。也就是「涨多了会回、跌多了会弹」的物极必反。它赚的不是公司变好的钱,也不是趋势延续的钱,而是市场短期情绪过度宣泄之后、价格往回修正的那一下。注意,这是个很短周期的效应,跟中长期的趋势是两码事。


### 2. 2. 反转为什么存在:过度反应 + 流动性提供

反转背后有两个原因。一是过度反应:一个利空消息出来,情绪化的散户会恐慌抛售,把价格砸到远低于它该有的水平,等情绪平复,价格自然往回弹。二是流动性提供的补偿:当所有人都在恐慌卖出时,敢于在那一刻接盘、提供流动性的人,事后会得到一个反弹作为回报。这两点决定了:越是散户多、越情绪化、越容易过度反应的票,短期反转越明显。


### 3. 3. 反转 vs 动量:同一批票,短期买输家、中期买赢家

这是这一节最烧脑也最关键的一点。同样一批股票,在短期(一两周)看,你应该买上周的输家、赌它反弹;在中期(半年到一年)看,你又应该买过去的赢家、赌它强者恒强。方向正好相反!这不矛盾,是因为时间尺度不同:短期是情绪的过度反应在修正,中期是趋势和基本面在延续。理解了这个,你就不会在错误的时间尺度上用错策略——拿着中期动量的逻辑去做短线、或者拿短期反转的逻辑去做波段,都会南辕北辙。


### 4. 4. A股短期反转的真相:大盘弱、小盘才强(诚实)

这里我必须诚实地讲清楚一个实测结论:短期反转在A股不是到处都灵。在流动性好的大盘股、白马股上,它非常弱、很不稳定——因为这些票被机构盯得紧、定价有效,不太会过度反应。它主要出现在小盘、高换手、散户密集的票上。所以你听到「超跌反弹」就无脑抄大盘白马,大概率会像老王一样不灵。反转因子有很挑剔的适用范围,这一点远比「它存不存在」更重要。


### 5. 5. 反转的更多形态:隔夜、日内、成交量

反转不止「周度」一种。研究还发现:隔夜常有反转(收盘到第2 天开盘),日内也常有反转(开盘到收盘);而且成交量越大的时候、反转往往越显著,因为大成交量往往意味着情绪宣泄得越彻底。这些更精细的反转,大多被高频和量化机构用算法去捕捉,普通散户很难靠手动操作吃到,了解即可,别轻易下场跟机器抢这几个基点。


## 实操:反转因子:短期买上周输家(周度反转)+ 跟中期动量方向相反

本节无外部数据,纯模拟/统计运算,国内国外都能跑。**直接 Run All** 看结果。

**依赖:** `pip install pandas numpy matplotlib akshare statsmodels scipy`

In [3]:
# day_072_reversal_factor.py — 反转因子:短期买上周输家(周度反转)+ 跟中期动量方向正好相反
# 真名上屏:baostock / query_history_k_data_plus → close(前复权收盘) / resample('W-FRI') 周度收益 / nsmallest 上周输家组 / nlargest 上周赢家组
# 核心类比:短期(1 周)股价像橡皮筋——上周被砸得最狠的票,下周往往「弹一下」回来;所以短期要「买输家」。
#          但把尺度拉长到几个月,强者恒强——过去半年涨得猛的继续猛(动量,买赢家)。
#          同一批票,短期买输家、中期买赢家,「输赢」含义在两个时间尺度上正好颠倒,这就是反转 vs 动量的分水岭。
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import baostock as bs


def _data_path(_name):
    # 铁律62:data/ 放在 notebook 文件夹里。仓库根(run_lab)存取 out/notebook/data/;
    # 原版 notebook 在 out/notebook/ 用 data/;中国版在 out/notebook/cn/ 用 ../data/
    from pathlib import Path as _P
    _here = _P.cwd()
    for _b in [_here / 'data', _here / '..' / 'data', _here / 'out' / 'notebook' / 'data', _here / '..' / '..' / 'data', _here / '..' / '..' / '..' / 'data']:
        if (_b / _name).exists():
            return str(_b / _name)
    if (_here / 'out' / 'notebook').exists():
        _t = _here / 'out' / 'notebook' / 'data'
    elif _here.name == 'cn':
        _t = _here / '..' / 'data'
    else:
        _t = _here / 'data'
    _t.mkdir(parents=True, exist_ok=True)
    return str(_t / _name)


pd.set_option('display.width', 160)
plt.rcParams['axes.unicode_minus'] = False

# ==== 标的池:~22 只小盘、高换手、散户密集的活跃题材股(短期反转在小盘散户票才明显)====
# 全部 2018 年前已上市;扩大票池(每组取约 1/3 ≈ 7 只)大幅降低小样本噪声。
# 短期反转是「流动性提供溢价」:散户在小盘票上过度反应,上周被砸狠了的下周常被买回来。
STOCKS = {
    # 软件/互联网/题材(散户高换手)
    '大智慧': 'sh.601519', '二三四五': 'sz.002195', '网宿科技': 'sz.300017',
    '银之杰': 'sz.300085', '信雅达': 'sh.600571', '国民技术': 'sz.300077',
    '朗科科技': 'sz.300042', '三泰控股': 'sz.002312',
    # 小盘消费/食品(散户偏爱)
    '全聚德': 'sz.002186', '西安饮食': 'sz.000721', '莲花健康': 'sh.600186',
    '海欣食品': 'sz.002702', '金健米业': 'sh.600127',
    # 小盘医药
    '京新药业': 'sz.002020', '福瑞股份': 'sz.300049',
    # 小盘化工/有色/材料(周期波动大)
    '沧州大化': 'sh.600230', '宏达股份': 'sh.600331', '金浦钛业': 'sz.000545',
    '秀强股份': 'sz.300160',
    # 小盘机械/汽配/电子
    '京威股份': 'sz.002662', '通达动力': 'sz.002576', '华资实业': 'sh.600191',
}
START, END = '2018-01-01', '2024-12-31'
CSV_PX = _data_path('D072_reversal_close.csv')   # 前复权收盘(算周度真实收益)


# ==== 0. 拉数据:前复权收盘(adjustflag='2')。load 优先 + fetch 兜底 + to_csv 保存(铁律62) ====
def _fetch():
    bs.login()
    px = {}
    for name, code in STOCKS.items():
        rs = bs.query_history_k_data_plus(code, 'date,close', start_date=START, end_date=END, frequency='d', adjustflag='2')
        r = []
        while rs.error_code == '0' and rs.next():
            r.append(rs.get_row_data())
        s = pd.DataFrame(r, columns=['date', 'close'])
        s['close'] = pd.to_numeric(s['close'], errors='coerce')
        s = s.set_index('date')
        px[name] = s['close']
    bs.logout()
    return pd.DataFrame(px)


if os.path.exists(CSV_PX):
    print('[数据] 从本地 CSV 读取 前复权收盘 ->', CSV_PX)
    px = pd.read_csv(CSV_PX, parse_dates=['date']).set_index('date')
else:
    print('[数据] baostock 拉取 %d 只票 前复权收盘 ...' % len(STOCKS))
    px = _fetch()
    px.index.name = 'date'
    px.to_csv(CSV_PX)               # 拉完保存 CSV(铁律62)
    print('[数据] 已存 CSV ->', CSV_PX)

px.index = pd.to_datetime(px.index)   # 两条路径(CSV/fetch)统一成日期索引
px = px.sort_index()

print('\n==== 反转因子:短期买上周输家(周度反转) + 跟中期动量方向相反 ====')
print('标的池 %d 只 · %s ~ %s' % (len(STOCKS), px.index[0].date(), px.index[-1].date()))


def ann_return(equity, n_periods, periods_per_year):
    # 年化收益(%):把总收益按「期数/每年期数」折算成每年
    e = pd.Series(equity).dropna()
    if len(e) < 2 or n_periods <= 0:
        return 0.0
    yrs = n_periods / periods_per_year
    return float((e.iloc[-1] ** (1.0 / yrs) - 1.0) * 100)


def maxdd(equity):
    # 最大回撤(%):从历史最高点跌下来最深的那一下
    e = pd.Series(equity).dropna()
    if len(e) < 2:
        return 0.0
    peak = e.cummax()
    return float((e / peak - 1.0).min() * 100)


# ==== 1. 周度重采样:每周五收盘价 -> 周收益 ====
weekly = px.resample('W-FRI').last()        # 每个自然周的周五收盘(最后一个交易日)
wret = weekly.pct_change()                  # 本周收益 = 本周五 / 上周五 - 1(周末即已知)
wfwd = wret.shift(-1)                        # 下一周收益(本周末建仓 -> 下周末收益)


def backtest_weekly(mode):
    # 每周末按「上周收益」排序,等权持有下一周。
    # mode: 'loser'=买上周跌最惨的输家组 / 'winner'=买上周涨最猛的赢家组 / 'base'=全池等权基准
    rets, dates = [], []
    for t in weekly.index:
        sig = wret.loc[t].dropna()          # 上周收益(刚刚实现,周末即已知)
        fwd = wfwd.loc[t]                    # 下周收益(待实现)
        n = len(sig)
        if n < 2:
            continue
        k = min(max(3, n // 3), n)           # 取约 1/3、至少 3 只
        if mode == 'loser':
            sel = sig.nsmallest(k).index     # 上周跌最惨的一组(输家)
        elif mode == 'winner':
            sel = sig.nlargest(k).index      # 上周涨最猛的一组(赢家)
        else:
            sel = sig.index                  # 基准:全池等权
        r = fwd[sel].mean()                  # 下周等权平均收益(mean 自动跳过停牌 NaN)
        if pd.isna(r):
            continue                         # 仅最后一周 fwd 全 NaN,跳过收尾
        rets.append(r)
        dates.append(t)
    eq = (1 + pd.Series(rets, index=dates)).cumprod()
    return eq


eq_loser = backtest_weekly('loser')
eq_winner = backtest_weekly('winner')
eq_base = backtest_weekly('base')
n_weeks = len(eq_loser)

print('\n[短期反转:每周买上周输家 vs 买上周赢家 vs 基准等权 — 同一段 %d 周]' % n_weeks)
for tag, eq in [('上周输家组(反转)', eq_loser), ('上周赢家组', eq_winner), ('基准等权', eq_base)]:
    tot = (eq.iloc[-1] - 1) * 100
    print('%-14s 总收益 %+7.1f%%   年化 %+6.1f%%   最大回撤 %6.1f%%' % (
        tag, tot, ann_return(eq.values, n_weeks, 52), maxdd(eq.values)))

_loser_tot = (eq_loser.iloc[-1] - 1) * 100
_winner_tot = (eq_winner.iloc[-1] - 1) * 100
_rev_gap = _loser_tot - _winner_tot
print('[短期反转效应] 上周输家组比上周赢家组多赚 %+.1f 个百分点 -> %s' % (
    _rev_gap, '输家组反弹跑赢赢家组,短期反转成立' if _rev_gap > 0 else '本池本段不显著,反转未跑出'))


# ==== 2. 上周收益 -> 下周收益:分档看反弹幅度(反转效应的直接证据) ====
# 汇总全部 (上周收益, 下周收益) 观测点,按上周收益从低到高分 5 档,看每档下周平均收益
obs_prev, obs_next = [], []
for t in weekly.index:
    sig = wret.loc[t].dropna()
    fwd = wfwd.loc[t]
    for name in sig.index:
        if not pd.isna(fwd.get(name, np.nan)):
            obs_prev.append(sig[name])
            obs_next.append(fwd[name])
obs = pd.DataFrame({'prev': obs_prev, 'next': obs_next})
obs['bucket'] = pd.qcut(obs['prev'], 5, labels=['最惨档', '偏弱档', '中间档', '偏强档', '最猛档'])
bucket_next = obs.groupby('bucket', observed=True)['next'].mean() * 100   # 每档下周平均收益(%)
bucket_cnt = obs.groupby('bucket', observed=True)['next'].size()

print('\n[上周收益分档 -> 下周平均收益](共 %d 个「票×周」观测点)' % len(obs))
for b in bucket_next.index:
    print('  上周%s  下周平均 %+6.2f%%   (样本 %d)' % (b, bucket_next[b], bucket_cnt[b]))
_worst_next = bucket_next['最惨档']
_best_next = bucket_next['最猛档']
print('[反弹幅度] 上周跌最惨的「最惨档」下周平均 %+.2f%%,上周涨最猛的「最猛档」下周平均 %+.2f%%,差 %+.2f 个百分点' % (
    _worst_next, _best_next, _worst_next - _best_next))
# 上周收益 vs 下周收益的回归斜率(负斜率 = 反转:上周越跌、下周越弹)
_slope, _intercept = np.polyfit(obs['prev'].values, obs['next'].values, 1)
_corr = float(obs['prev'].corr(obs['next']))
print('[负相关证据] 上周收益 vs 下周收益 回归斜率 = %.3f(负数=反转),相关系数 = %.3f' % (_slope, _corr))


# ==== 3. 中期动量:每月买过去半年赢家(跟短期反转方向正好相反) ====
monthly = px.resample('ME').last()          # 月末收盘
mret = monthly.pct_change()                 # 本月收益
mfwd = mret.shift(-1)                        # 下月收益(本月末建仓 -> 下月末收益)
# 过去 6 个月动量(跳过最近 1 个月,经典 6-1,避免短期反转污染)
mom = monthly.shift(1) / monthly.shift(7) - 1


def backtest_monthly(mode):
    # 每月末按「过去半年涨幅」排序,等权持有下一月。
    # mode: 'winner'=买过去半年赢家(动量) / 'loser'=买过去半年输家 / 'base'=全池等权
    rets, dates = [], []
    for t in monthly.index:
        sig = mom.loc[t].dropna()
        fwd = mfwd.loc[t]
        n = len(sig)
        if n < 2:
            continue
        k = min(max(3, n // 3), n)
        if mode == 'winner':
            sel = sig.nlargest(k).index      # 过去半年涨最猛(动量买赢家)
        elif mode == 'loser':
            sel = sig.nsmallest(k).index
        else:
            sel = sig.index
        r = fwd[sel].mean()
        if pd.isna(r):
            continue
        rets.append(r)
        dates.append(t)
    eq = (1 + pd.Series(rets, index=dates)).cumprod()
    return eq


eq_mom_win = backtest_monthly('winner')     # 中期动量:买过去半年赢家
eq_mom_lose = backtest_monthly('loser')     # 中期反向:买过去半年输家(做对照)
n_months = len(eq_mom_win)
_mom_win_tot = (eq_mom_win.iloc[-1] - 1) * 100
_mom_lose_tot = (eq_mom_lose.iloc[-1] - 1) * 100
_mom_gap = _mom_win_tot - _mom_lose_tot

print('\n[中期动量:每月买过去半年赢家 vs 买过去半年输家 — 同一段 %d 个月]' % n_months)
for tag, eq in [('过去半年赢家组(动量)', eq_mom_win), ('过去半年输家组', eq_mom_lose)]:
    tot = (eq.iloc[-1] - 1) * 100
    print('%-18s 总收益 %+7.1f%%   年化 %+6.1f%%' % (tag, tot, ann_return(eq.values, n_months, 12)))
print('[中期动量效应] 过去半年赢家组比输家组多赚 %+.1f 个百分点 -> %s' % (
    _mom_gap, '强者恒强,中期动量买赢家成立' if _mom_gap > 0 else '本池本段动量不显著'))

print('\n[★ 反转 vs 动量:方向正好相反 ★]')
print('  短期(1 周):买「上周输家」比买「上周赢家」多赚 %+.1f 个百分点 -> 短期买输家' % _rev_gap)
print('  中期(6 月):买「过去半年赢家」比买「过去半年输家」多赚 %+.1f 个百分点 -> 中期买赢家' % _mom_gap)
print('  同一批票,短期「输赢」和中期「输赢」含义颠倒:短期赌反弹买输家、中期赌趋势买赢家。')


# ==== 4. 三张图 ====
# 图1:短期反转(买上周输家) vs 买上周赢家 vs 基准 三条周度净值,终值标注
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(eq_loser.index, eq_loser.values, lw=2, c='#2e7d32', label='上周输家组(短期反转)')
ax.plot(eq_winner.index, eq_winner.values, lw=2, c='#c62828', label='上周赢家组')
ax.plot(eq_base.index, eq_base.values, lw=1.6, c='#777', ls='--', label='基准等权')
ax.axhline(1.0, c='k', lw=.8)
for eq, c in [(eq_loser, '#2e7d32'), (eq_winner, '#c62828'), (eq_base, '#777')]:
    ax.text(eq.index[-1], eq.iloc[-1], ' %.2f' % eq.iloc[-1], color=c, fontweight='bold', va='center')
_t1 = '输家组反弹跑赢赢家组,短期反转成立' if _rev_gap > 0 else '本池本段反转不显著(输家组未跑赢)'
ax.set_title('短期反转:每周买「上周输家」vs「上周赢家」vs 基准 —— %s(终值已标注)' % _t1)
ax.set_ylabel('净值(起点=1,周度调仓)')
ax.legend(loc='upper left')
ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig('chart_01.png', dpi=110)
plt.close()

# 图2:短期反转策略(周度买上周输家) vs 中期动量策略(月度买过去半年赢家) 叠一起,凸显方向相反
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(eq_loser.index, eq_loser.values, lw=2, c='#1565c0', label='短期反转:每周买「上周输家」')
ax.plot(eq_mom_win.index, eq_mom_win.values, lw=2, c='#e65100', label='中期动量:每月买「过去半年赢家」')
ax.axhline(1.0, c='k', lw=.8)
for eq, c in [(eq_loser, '#1565c0'), (eq_mom_win, '#e65100')]:
    ax.text(eq.index[-1], eq.iloc[-1], ' %.2f' % eq.iloc[-1], color=c, fontweight='bold', va='center')
ax.set_title('同一批票,短期买输家、中期买赢家:反转与动量「输赢」含义正好颠倒')
ax.set_ylabel('净值(起点=1)')
ax.legend(loc='upper left')
ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig('chart_02.png', dpi=110)
plt.close()

# 图3:上周收益分 5 档 -> 下周平均收益柱状(负相关 = 上周跌得越狠、下周反弹越多)
fig, ax = plt.subplots(figsize=(11, 6))
labels = list(bucket_next.index)
vals = bucket_next.values
colors = ['#2e7d32' if v >= 0 else '#c62828' for v in vals]
ax.bar(labels, vals, color=colors)
ax.axhline(0, c='k', lw=.8)
for i, v in enumerate(vals):
    ax.text(i, v + (0.03 if v >= 0 else -0.06), '%+.2f%%' % v, ha='center', fontsize=9, fontweight='bold')
_t3 = ('斜率 %.3f<0:跌得越狠下周反弹越多=反转' % _slope) if _slope < -0.01 else ('斜率 %.3f≈0:单股层面反转信号很弱(A股短期反转不稳)' % _slope)
ax.set_title('上周收益分档 → 下周平均收益(%s)' % _t3)
ax.set_ylabel('下周平均收益(%)')
ax.set_xlabel('按上周收益从低到高分档')
ax.grid(alpha=.3, axis='y')
fig.tight_layout()
fig.savefig('chart_03.png', dpi=110)
plt.close()

print('\n[done] 反转因子:短期买上周输家(周度反转) + 跟中期动量方向相反 —— 3 图已出')


[数据] 从本地 CSV 读取 前复权收盘 -> /mnt/d/huizi_ai_project/ai_course_video/out/notebook/cn/../data/D072_reversal_close.csv

==== 反转因子:短期买上周输家(周度反转) + 跟中期动量方向相反 ====
标的池 22 只 · 2018-01-02 ~ 2024-12-31

[短期反转:每周买上周输家 vs 买上周赢家 vs 基准等权 — 同一段 340 周]
上周输家组(反转)      总收益   +40.2%   年化   +5.3%   最大回撤  -44.2%
上周赢家组          总收益   +31.9%   年化   +4.3%   最大回撤  -42.7%
基准等权           总收益   +59.0%   年化   +7.3%   最大回撤  -39.1%
[短期反转效应] 上周输家组比上周赢家组多赚 +8.4 个百分点 -> 输家组反弹跑赢赢家组,短期反转成立

[上周收益分档 -> 下周平均收益](共 7480 个「票×周」观测点)
  上周最惨档  下周平均  +0.34%   (样本 1496)
  上周偏弱档  下周平均  +0.11%   (样本 1496)
  上周中间档  下周平均  -0.05%   (样本 1496)
  上周偏强档  下周平均  +0.26%   (样本 1496)
  上周最猛档  下周平均  +0.40%   (样本 1496)
[反弹幅度] 上周跌最惨的「最惨档」下周平均 +0.34%,上周涨最猛的「最猛档」下周平均 +0.40%,差 -0.06 个百分点
[负相关证据] 上周收益 vs 下周收益 回归斜率 = 0.048(负数=反转),相关系数 = 0.048

[中期动量:每月买过去半年赢家 vs 买过去半年输家 — 同一段 76 个月]
过去半年赢家组(动量)        总收益  +172.7%   年化  +17.2%
过去半年输家组            总收益  +100.4%   年化  +11.6%
[中期动量效应] 过去半年赢家组比输家组多赚 +72.3 个百分点 -> 强者恒强,中期动量买赢家成立

[★ 反转 vs 动量:方向正好相反 ★]
  短期(1 周)

## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| 中期动量强·短期反转弱 | N/A | 2018到2024这7 年,同一批22只小盘票:中期动量(每月买过去半年的赢家)总收益+172.7%、终值2.73,赢家比输家多赚+72.3个点,强者恒强很稳;而短期反转(每周买上周的输家)总收益只有+40.2%、终值1.40,仅小幅跑赢上周赢家+8.4个点,而且两个都没跑赢基准+59%。诚实地说,A股真正稳健的是中期动量,短期反转弱得多。 |
| 短期反转很弱很不稳(诚实) | N/A | 组合层面每周买上周输家+40.2%比买上周赢家+31.9%略高,方向对;但单股层面把上周收益分5档看下周,呈U形——最惨档下周+0.34%、最猛档反而+0.40%最高,回归斜率只有+0.048几乎为零。短期反转在中大盘股上几乎不显著,我们换了22只小盘散户活跃票才勉强为正,可见它有很挑剔的适用范围。 |
| 反转vs动量方向相反·核心 | N/A | 同一批票,短期(1周)该买上周的输家、赌超跌反弹,中期(半年)却该买过去的赢家、赌强者恒强,方向正好相反。不矛盾,是时间尺度不同:短期是情绪过度反应在修正,中期是趋势在延续。理解这个能避免在错误的时间尺度上用错策略,拿中期动量逻辑去做短线、或拿短期反转逻辑去做波段,都会南辕北辙。 |
| 为什么大盘弱小盘强 | N/A | 短期反转的根在过度反应和流动性提供:情绪化散户把跌狠的票砸到不合理低位,敢接盘的人赚个反弹。所以越是散户多、越情绪化的小盘高换手票反转越明显;流动性好、被机构盯紧、定价有效的大盘白马几乎不反应。老王专抄大盘白马超跌反弹不灵,正是挑错了票池。而且反转高换手,佣金印花税价差极易吃光收益。 |


## 常见坑

### ⚠ 01. 把反转当成万能抄底口诀

「跌多了就会反弹」只在很短周期、特定票池里成立。把它当成万能抄底,看到什么票跌得多都去接,很容易接在半山腰、一路套下去。反转有严格的时间和标的适用范围。

### ⚠ 02. 在下跌趋势里硬抄

短期反转赌的是情绪过度宣泄后的修正,可如果一只票是基本面恶化、走在长期下跌趋势里,那它的「跌」不是过度反应,而是价值在崩塌。这时候抄底就是接飞刀,越抄越亏。要分清是情绪超跌还是趋势崩塌。

### ⚠ 03. 拿大盘白马做短期反转

像老王一样,在流动性好、定价有效的大盘股上做短期反转,大概率不灵。反转主要在小盘、高换手、散户密集的票上才明显。挑错池子,策略再对也白搭。

### ⚠ 04. 忽略高换手带来的交易成本

短期反转要求频繁调仓(周度甚至日度),换手率极高。佣金、印花税、买卖价差累加起来非常可观,很可能把那点反弹收益全吃掉。算清交易成本,反转策略对成本极其敏感。

### ⚠ 05. 把价值低估当短期反转

价值因子的「便宜」是基本面层面的长期低估,反转的「超跌」是情绪层面的短期错杀,两者完全不同。把一只长期低估值的票当成短期反转标的去做波段,时间尺度和逻辑都错了。

## 实战 SOP · 用反转因子 — 几条铁规矩

1. 先想清楚反转在赌什么:赌短期情绪过度反应之后的价格修正,是很短周期的效应,跟中长期趋势是两码事。
2. 认清适用范围:短期反转在大盘白马上很弱,主要在小盘、高换手、散户密集的票上才明显,挑错池子必不灵。
3. 分清超跌和崩塌:情绪砸出来的超跌才会反弹,基本面恶化的趋势性下跌是接飞刀,别硬抄。
4. 务必算清交易成本:反转高换手,佣金、印花税、价差极易吃光收益,对成本极其敏感。
5. 记住和动量的关系:短期买输家、中期买赢家,方向相反、各管各的时间尺度,别用错。

> 把这段打印贴在你电脑边。

## 总结 · 你应该带走的

2. 反转因子=短期内(1到4周)跌最惨的超跌反弹、涨最猛的回落,赌情绪过度宣泄后价格往回修正。
3. 为什么有反转:散户过度反应把价格砸过头,加上敢接盘提供流动性的人赚个反弹补偿。
4. 诚实结果:A股短期反转很弱很不稳——组合层面每周买上周输家+40.2%仅小幅跑赢赢家+8.4点,单股回归斜率+0.048≈0,都没跑赢基准+59%。
5. 真正稳健的是中期动量:每月买过去半年赢家终值2.73、多赚+72.3点,远强于短期反转终值1.40。
6. 反转和动量方向正好相反:同一批票短期买输家、中期买赢家,时间尺度不同、输赢颠倒,别用错尺度。
7. 反转挑票池:大盘白马很弱、小盘高换手散户票才明显;还高换手、对交易成本极敏感,绝不能硬抄。

## 自测题

**Q1.** 反转因子和上一节的动量因子,说的好像正相反(一个跌多了反弹、一个强者恒强),它们到底矛盾不矛盾?用「时间尺度」这个词解释清楚。

**Q2.** 老王专抄跌得最惨的大盘白马,为什么短期反转对他不灵?反转因子在什么样的票池里才明显、为什么?

**Q3.** 短期反转要频繁调仓、换手率很高,这会带来什么问题?为什么说反转策略对交易成本「极其敏感」?

把答案写下来,3 天后再回看。

## 下一节预告

**Day 073 · 低波动因子** (Low-Vol Anomaly)

这一节我们看清了短期反转的真相:它存在、但很挑剔,还跟中期动量方向相反。下一节我们讲一个更反直觉、也更适合普通散户的因子:低波动因子。课本告诉你高风险高回报,可现实里偏偏有个「低波异常」——那些波动小、走得稳的票,风险调整之后反而更划算。越稳越赚,这到底是怎么回事?下一节揭晓。

## 推荐阅读

- Narasimhan Jegadeesh《Evidence of Predictable Behavior of Security Returns》(1990/JF) — 短期(月度)反转的奠基研究,首次系统证明上期输家下期反弹
- Bruce Lehmann《Fads, Martingales, and Market Efficiency》(1990/QJE) — 周度反转的经典文献,把短期反转和流动性提供的补偿联系起来
- Werner De Bondt, Richard Thaler《Does the Stock Market Overreact?》(1985/JF) — 行为金融名篇:投资者过度反应是反转效应的心理根源,反转研究的源头
- Mark Grinblatt, Tobias Moskowitz《Predicting Stock Price Movements from Past Returns》(2004/JFE) — 系统梳理短期反转与中期动量如何在不同时间尺度上共存,本节核心对照
- 丹尼尔·卡尼曼《思考,快与慢》(2012/中信出版社) — 用大白话讲透人的「过度反应」「损失厌恶」等心理偏差,看懂反转因子背后的人性